# 01b — EDA tổng hợp trên DEV

Notebook này chỉ trình bày các tổng hợp do `scripts/build_eda_dev_artifacts.py` tạo từ DuckDB DEV read-only và mẫu nhãn DEV lịch sử. Cửa sổ đơn là `[2024-04-01, 2025-10-22)`. Không đọc snapshot raw, feature scoring của cohort 5.169 khách, nhãn/hành vi test, hoặc bất kỳ khóa HMAC nào.

Các phân phối là mô tả. Chúng không chứng minh mùa vụ, nguyên nhân mua lại, lợi nhuận hay hiệu quả can thiệp. Các nhóm nhỏ được nêu kèm số mẫu. Xem định nghĩa cohort tại `docs/cohort_definitions.md` và lý do metric tại `docs/metric_rationale.md`.

## Quan sát và hệ quả khi đọc kết quả

1. Có 6.400 đơn status 3 từ tháng 4/2024 đến trước 22/10/2025. Tháng 10/2024 có 619 đơn, tháng thấp nhất trong cửa sổ là tháng 4/2025 với 217; tháng 10/2025 chỉ có dữ liệu tới ngày 21 (543 đơn). Vì vậy không gắn nhãn mùa vụ hay diễn giải đỉnh tháng nếu chưa có kiểm chứng nghiệp vụ.
2. Có 5.174 khách có mã trên các đơn status 3; 4.526 (87,5%) có một đơn, 648 (12,5%) có từ hai đơn. Các mô tả khách hàng bị chi phối bởi khách mua một lần, nên nhóm lặp lại có quy mô nhỏ hơn nhiều.
3. Trong 6.400 đơn hợp lệ có 286 đơn thiếu mã khách. Không thể nối các đơn này vào lịch sử khách; các cohort customer-level vì thế không đồng nghĩa toàn bộ đơn.
4. Median giỏ là 4 product phân biệt, p90 là 7. Có 361/6.400 đơn (5,6%) có 0 hoặc 1 product phân biệt; luật kết hợp chỉ áp dụng cho phần giỏ thỏa điều kiện, không đại diện cho mọi đơn.
5. DEV có 3.508 mẫu nhãn, trong đó 260 dương (7,41%). Accuracy có thể trông cao nếu chỉ đoán lớp âm; cần đọc AP/PR cùng prevalence và xem precision/recall tại ngưỡng đã khóa.
6. Trong mẫu nhãn, nhóm có một đơn trước T có 225/3.329 lần mua lại (6,76%); nhóm có từ hai đơn có 35/179 (19,55%). Đây là liên hệ mô tả trên nhóm 2+ nhỏ, không phải tác động của việc mua lặp và không thay thế kiểm định ngoài mẫu.
7. Tỷ lệ nhãn theo recency không tăng/giảm đều: lần lượt 10,0% (0–7 ngày), 6,6% (8–14), 6,6% (15–21), 7,6% (22–28), 5,4% (29+). Không nên áp một quy tắc tuyến tính đơn giản; giữ đánh giá theo thời gian và xem cỡ mẫu từng khoảng.
8. AOV đặc trưng có median 280.000 VND, p95 1.075.000 VND và p99 khoảng 2.956.000 VND. Đuôi phải dài khiến median/quantile và thang log dễ đọc hơn giá trị trung bình đơn lẻ. Đây là giá trị theo quy tắc tiền hiện có, không phải đối soát kế toán độc lập.
9. Tần suất status 3 theo tháng dao động rõ (619 ở 10/2024 so với 217 ở 04/2025), nhưng artifact chỉ có ngày và status hiện tại; không có giải thích chiến dịch/tồn kho/nguyên nhân. Dùng như giả thuyết kiểm tra sau này, không kết luận nhân quả.
10. Historical T của nhãn DEV trải từ 2024-05-01 đến 2025-04-01 qua 12 mốc. Kết quả phản ánh thiết kế nhãn N=180 của pipeline qua nhiều giai đoạn, không phải tỷ lệ quay lại của toàn bộ snapshot.

## Nguồn và tái tạo

Nguồn: `data/restricted/xoaif_dev.duckdb` (kết nối read-only, truy vấn DEV trước T), `outputs/tables/day2_label_samples_primary.csv` (DEV historical labels), `outputs/metrics/eda_dev_summary.json` và năm hình `outputs/figures/eda_dev_*.png`. Không chạy notebook này trong giai đoạn hậu mở test; để xem artifact đã lưu, chạy duy nhất cell trình bày bên dưới nếu môi trường được dùng cho đọc báo cáo.

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display
import pandas as pd

CWD = Path.cwd()
ROOT = CWD.parent if CWD.name == 'notebooks' else CWD
summary = json.loads((ROOT / 'outputs/metrics/eda_dev_summary.json').read_text(encoding='utf-8'))
assert summary['scope']['period'] == '[2024-04-01, 2025-10-22)'
assert summary['scope']['test_data_read_by_this_build'] is False
display(pd.DataFrame(summary['monthly_valid_orders']))
display(pd.DataFrame(summary['dev_label_samples']['frequency_bands']))
for name in ('eda_dev_monthly_orders.png', 'eda_dev_basket_size.png', 'eda_dev_customer_frequency.png', 'eda_dev_label_rate_by_frequency.png', 'eda_dev_aov_distribution.png'):
    display(Image(filename=str(ROOT / 'outputs/figures' / name)))